In [ ]:
conn = psycopg2.connect(
    host=PG_HOST, port=PG_PORT, user=PG_USER, password=PG_PASSWORD, dbname=PG_DB
)

# Cell 5: Compute Column Statistics
from pipeline.dbt_generator import compute_column_stats

print("Computing column statistics...")
all_stats = {}  # table_name -> col_name -> stats

for table in table_names:
    cols = table_columns.get(table, [])
    if cols:
        stats = compute_column_stats(conn, table, cols)
        all_stats[table] = stats
        print(f"  {table}: {len(cols)} columns analyzed")

def _format_values(items, limit=3):
    if not items:
        return "N/A"
    return ", 
{item.get('value')} ({item.get('cnt')})" for item in items[:limit])

# Build summary DataFrame
rows = []
for table, col_stats in all_stats.items():
    for col_name, s in col_stats.items():
        rows.append(
            {
                "table": table,
                "column": col_name,
                "null_rate": round(s.get("null_rate", 0), 4),
                "non_null_count": s.get("non_null_count", 0),
                "uniqueness_score": round(s.get("uniqueness_score", 0), 4),
                "distinct_count": s.get("distinct_count", 0),
                "zero_count": s.get("zero_count"),
                "min_val": s.get("min_val"),
                "max_val": s.get("max_val"),
                "mean_val": s.get("mean_val"),
                "median_val": s.get("median_val"),
                "p75_val": s.get("p75_val"),
                "min_length": s.get("min_length"),
                "max_length": s.get("max_length"),
                "avg_length": s.get("avg_length"),
                "true_count": s.get("true_count"),
                "false_count": s.get("false_count"),
                "top_values": _format_values(s.get("top_values", [])),
                "sample_values": ", 
sample_values", [])[:5]) or "N/A",
                "total_count": s.get("total_count", 0),
            }
        )

stats_df = pd.DataFrame(rows).sort_values("null_rate", ascending=False)
print(f"\nColumn stats summary (sorted by null_rate):")
display(stats_df.head(30))
